# 🚀 EPRBench (Lightning + Hydra) — Google Colab 一键运行笔记本

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/trimpouo/eprbench_lightning/blob/main/notebooks/colab_eprbench.ipynb)

本 Notebook 专为在 **Google Colab** 中运行 [`trimpouo/eprbench_lightning`](https://github.com/trimpouo/eprbench_lightning) 设计，完整覆盖：
1. **GPU 环境检查与 Google Drive 挂载**（自动对接 `我的云端硬盘 > datasets > EPRBench`）
2. **拉取 GitHub 仓库与安装依赖**（自动适配 Colab Python 3.11/3.12 的 `faiss-cpu` 与 NumPy 2.x 兼容性）
3. **拉取预训练权重与解压数据集**（将 `model/dinov2_vitb14_pretrain.pth`、`model/SGVPR.pth` 与 `EventVPR.7z` 解压至 Colab 本地高速 SSD，避免云盘小文件读取瓶颈）
4. **官方权重 `SGVPR.pth` 推理评估与 Top-K 可视化**
5. **PyTorch Lightning + Hydra 模型训练**（支持快速 Smoke Test、Baseline 训练、SACF 实验等）
6. **训练权重评估 (`src/eval.py`) 与 Google Drive 自动备份 (`logs_backup`)**

## 1. 检查 GPU 环境并挂载 Google Drive

运行前请确保在 Colab 菜单中选择：**代码执行程序 (Runtime) -> 更改运行时类型 (Change runtime type) -> 硬件加速器: GPU (T4 / L4 / A100)**。

In [ ]:
import os
import shutil
import subprocess
from pathlib import Path
import torch

os.environ["PYTHONHASHSEED"] = "0"

print(f"PyTorch version : {torch.__version__}")
print(f"CUDA available  : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
    print(f"GPU Device      : {gpu_name} ({vram_gb:.2f} GB VRAM)")
else:
    print("⚠️ 警告: 当前未检测到 GPU！请前往「代码执行程序 -> 更改运行时类型」开启 GPU。")

# 挂载 Google Drive
try:
    from google.colab import drive
    drive.mount("/content/drive")
except ImportError:
    print("当前不在 Google Colab 环境中，跳过 drive.mount。")

# 云盘路径配置（对应：我的云端硬盘 > datasets > EPRBench）
GDRIVE_EPRBENCH_DIR = Path("/content/drive/MyDrive/datasets/EPRBench")
GDRIVE_MODEL_DIR = GDRIVE_EPRBENCH_DIR / "model"
GDRIVE_ARCHIVE_PATH = GDRIVE_EPRBENCH_DIR / "EventVPR.7z"
GDRIVE_LOGS_BACKUP = GDRIVE_EPRBENCH_DIR / "logs_backup"

if GDRIVE_EPRBENCH_DIR.exists():
    print(f"\n✅ 成功找到云盘目录: {GDRIVE_EPRBENCH_DIR}")
    for item in sorted(GDRIVE_EPRBENCH_DIR.iterdir()):
        if item.is_file():
            size_gb = item.stat().st_size / (1024 ** 3)
            print(f"  📄 {item.name:<28} ({size_gb:.2f} GB)")
        else:
            print(f"  📁 {item.name}/")
else:
    raise FileNotFoundError(
        f"未找到 {GDRIVE_EPRBENCH_DIR}，请确认 Google Drive 是否已挂载且路径为 MyDrive/datasets/EPRBench"
    )

## 2. 克隆 GitHub 仓库并安装项目依赖

- 自动从 `https://github.com/trimpouo/eprbench_lightning.git` 拉取最新代码到 `/content/eprbench_lightning`
- Colab 默认 Python 3.11/3.12 下 PyPI 无 `faiss-gpu` 预编译包，而本项目仅使用 `faiss.IndexFlatL2`（CPU 索引），此处自动安装 `faiss-cpu` 并补齐所需依赖
- 自动修复 NumPy 2.x 中 `np.in1d` 弃用/移除的兼容性问题

In [ ]:
REPO_URL = "https://github.com/trimpouo/eprbench_lightning.git"
PROJECT_ROOT = Path("/content/eprbench_lightning")

if not PROJECT_ROOT.exists():
    print(f"正在克隆仓库 {REPO_URL} ...")
    subprocess.run(["git", "clone", REPO_URL, str(PROJECT_ROOT)], check=True)
else:
    print(f"仓库已存在 ({PROJECT_ROOT})，正在拉取最新更改...")
    subprocess.run(["git", "-C", str(PROJECT_ROOT), "pull"], check=False)

os.chdir(PROJECT_ROOT)
os.environ["PROJECT_ROOT"] = str(PROJECT_ROOT)
print(f"当前工作目录: {Path.cwd()}")

# 安装 Colab 缺失的核心依赖（保留 Colab 预装的 CUDA PyTorch，使用 faiss-cpu 替代无法在 Py3.11+ pip 安装的 faiss-gpu）
pip_packages = [
    "lightning>=2.0.0",
    "torchmetrics>=0.11.4",
    "hydra-core==1.3.2",
    "hydra-colorlog==1.2.0",
    "hydra-optuna-sweeper==1.2.0",
    "open_clip_torch>=2.20.0",
    "pytorch-metric-learning>=2.3.0",
    "faiss-cpu>=1.7.4",
    "PyWavelets>=1.4.1",
    "pytorch-wavelets>=1.3.0",
    "rootutils",
    "rich",
    "albumentations>=1.3.0",
    "timm>=1.0.0",
]
print("正在安装依赖包...")
subprocess.run(["pip", "install", "-q"] + pip_packages, check=True)

# 自愈检查 1: 若远端分支缺少 src/models/components/mods/temporal_event_aggregator.py 则自动补齐
tefa_path = PROJECT_ROOT / "src/models/components/mods/temporal_event_aggregator.py"
if not tefa_path.exists():
    tefa_path.write_text(
        '"""Temporal Event Frame Aggregator (TEFA) module."""\n\n'
        "import torch\nimport torch.nn as nn\nimport torch.nn.functional as F\n\n\n"
        "class TemporalEventFrameAggregator(nn.Module):\n"
        "    def __init__(self, channels: int = 768, reduction: int = 8) -> None:\n"
        "        super().__init__()\n"
        "        hidden = max(16, channels // reduction)\n"
        "        self.score_net = nn.Sequential(\n"
        "            nn.LayerNorm(channels), nn.Linear(channels, hidden), nn.GELU(), nn.Linear(hidden, 1)\n"
        "        )\n"
        "        self.res_scale = nn.Parameter(torch.zeros(1))\n\n"
        "    def forward(self, x: torch.Tensor) -> torch.Tensor:\n"
        "        if x.dim() == 5:\n"
        "            logits = self.score_net(x.mean(dim=(-2, -1)))\n"
        "            weights = F.softmax(logits, dim=1).unsqueeze(-1).unsqueeze(-1)\n"
        "            attn_out = (x * weights).sum(dim=1)\n"
        "            mean_out = x.mean(dim=1)\n"
        "            return mean_out + self.res_scale * (attn_out - mean_out)\n"
        "        if x.dim() == 3:\n"
        "            weights = F.softmax(self.score_net(x), dim=1)\n"
        "            attn_out = (x * weights).sum(dim=1)\n"
        "            mean_out = x.mean(dim=1)\n"
        "            return mean_out + self.res_scale * (attn_out - mean_out)\n"
        "        raise ValueError(f'Expected 3D or 5D tensor, got {x.shape}')\n",
        encoding="utf-8",
    )
    print("已自动补齐缺失的 src/models/components/mods/temporal_event_aggregator.py")

# 自愈检查 2: 兼容 NumPy 2.x，将源码中的 np.in1d 替换为 np.isin
for rel_path in ["src/models/eprbench_module.py", "src/visualize_server.py"]:
    fpath = PROJECT_ROOT / rel_path
    if fpath.exists():
        text = fpath.read_text(encoding="utf-8")
        if "np.in1d" in text:
            fpath.write_text(text.replace("np.in1d", "np.isin"), encoding="utf-8")
            print(f"已自动替换 {rel_path} 中的 np.in1d -> np.isin（兼容 NumPy 2.x）")

print("✅ 仓库与依赖准备完成！")

## 3. 从 Google Drive 拉取模型权重并解压 `EventVPR.7z`

> 💡 **为什么要解压到 `/content/eprbench_lightning/data/`？**
> EventVPR 包含约 6.5 万张事件帧图片和场景文本描述。如果直接在 `/content/drive` 云盘挂载目录上读取海量小文件，Colab 的 FUSE 文件系统会产生严重的 I/O 阻塞。将 `EventVPR.7z` 直接解压到 Colab 本地 SSD 只需约 1~2 分钟，后续训练与评估速度可提升数十倍。

In [ ]:
LOCAL_MODEL_DIR = PROJECT_ROOT / "model"
LOCAL_DATA_DIR = PROJECT_ROOT / "data"
LOCAL_EVENTVPR_DIR = LOCAL_DATA_DIR / "EventVPR"

LOCAL_MODEL_DIR.mkdir(parents=True, exist_ok=True)
LOCAL_DATA_DIR.mkdir(parents=True, exist_ok=True)

# 1. 拷贝预训练模型权重 (dinov2_vitb14_pretrain.pth & SGVPR.pth)
for weight_name in ["dinov2_vitb14_pretrain.pth", "SGVPR.pth"]:
    src_weight = GDRIVE_MODEL_DIR / weight_name
    dst_weight = LOCAL_MODEL_DIR / weight_name
    if dst_weight.exists() and dst_weight.stat().st_size > 1024 * 1024:
        print(f"✅ 权重已存在，跳过复制: {dst_weight} ({dst_weight.stat().st_size / (1024**2):.1f} MB)")
    elif src_weight.exists():
        print(f"⏳ 正在从云盘复制 {weight_name} 到本地...")
        shutil.copy2(src_weight, dst_weight)
        print(f"✅ 复制完成: {dst_weight} ({dst_weight.stat().st_size / (1024**2):.1f} MB)")
    elif weight_name == "dinov2_vitb14_pretrain.pth":
        print("⚠️ 云盘未找到 dinov2_vitb14_pretrain.pth，正在从官方源下载...")
        subprocess.run(
            [
                "wget",
                "-q",
                "--show-progress",
                "https://dl.fbaipublicfiles.com/dinov2/dinov2_vitb14/dinov2_vitb14_pretrain.pth",
                "-O",
                str(dst_weight),
            ],
            check=True,
        )
    else:
        print(f"⚠️ 未找到 {src_weight}")

# 2. 解压 EventVPR.7z 到本地 data/ 目录
if (LOCAL_EVENTVPR_DIR / "train").exists() and (LOCAL_EVENTVPR_DIR / "test").exists():
    print(f"✅ 数据集已解压就绪: {LOCAL_EVENTVPR_DIR}")
else:
    assert GDRIVE_ARCHIVE_PATH.exists(), f"未找到压缩包: {GDRIVE_ARCHIVE_PATH}"
    if shutil.which("7z") is None:
        subprocess.run(["apt-get", "update", "-qq"], check=False)
        subprocess.run(["apt-get", "install", "-y", "-qq", "p7zip-full"], check=True)

    print(f"⏳ 正在解压 {GDRIVE_ARCHIVE_PATH} 到 {LOCAL_DATA_DIR}（约需 1~3 分钟，请稍候）...")
    subprocess.run(
        ["7z", "x", str(GDRIVE_ARCHIVE_PATH), f"-o{LOCAL_DATA_DIR}", "-aos", "-bso0", "-bsp1"],
        check=True,
    )

    # 自动规范化目录层级：兼容压缩包顶层含或不含 EventVPR/ 的情况
    if not (LOCAL_EVENTVPR_DIR / "train").exists():
        if (LOCAL_DATA_DIR / "train").exists() and (LOCAL_DATA_DIR / "test").exists():
            LOCAL_EVENTVPR_DIR.mkdir(parents=True, exist_ok=True)
            for sub in ["train", "val", "test", "scene_descriptions"]:
                if (LOCAL_DATA_DIR / sub).exists():
                    shutil.move(str(LOCAL_DATA_DIR / sub), str(LOCAL_EVENTVPR_DIR / sub))
        elif (LOCAL_EVENTVPR_DIR / "EventVPR" / "train").exists():
            nested = LOCAL_EVENTVPR_DIR / "EventVPR"
            for item in nested.iterdir():
                shutil.move(str(item), str(LOCAL_EVENTVPR_DIR / item.name))
            nested.rmdir()

    print(f"✅ 数据集解压完成: {LOCAL_EVENTVPR_DIR}")

# 3. 校验数据集结构与样本数量
for split in ["train", "test"]:
    db_dir = LOCAL_EVENTVPR_DIR / split / "database"
    q_dir = LOCAL_EVENTVPR_DIR / split / "queries"
    n_db = len(list(db_dir.glob("@*"))) if db_dir.exists() else 0
    n_q = len(list(q_dir.glob("@*"))) if q_dir.exists() else 0
    print(f"  📊 [{split:<5}] database 场景数: {n_db:<5} | queries 场景数: {n_q}")

desc_dir = LOCAL_EVENTVPR_DIR / "scene_descriptions"
n_txt = len(list(desc_dir.glob("*.txt"))) if desc_dir.exists() else 0
print(f"  📝 [scene_descriptions] 文本描述文件数: {n_txt}")

## 4. 快速验证 I：使用官方权重 (`model/SGVPR.pth`) 进行测试集评估与可视化

> 注意：`model/SGVPR.pth` 是官方原始脚本导出的字典格式权重（包含 `model_state_dict` 与 `text_encoder_state_dict`），不是 PyTorch Lightning 的 `.ckpt` 文件。下面直接加载该权重，在 EventVPR 测试集上提取特征、运行 FAISS 检索并输出 Recall@1/5/10/100 及可视化结果。

In [ ]:
import random
import sys
import faiss
import hydra
import matplotlib.pyplot as plt
import numpy as np
import rootutils
from omegaconf import OmegaConf
from PIL import Image
from tqdm.auto import tqdm

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
rootutils.setup_root(str(PROJECT_ROOT), indicator=".project-root", pythonpath=True)

from src.data.eprbench_datamodule import EPRBenchDataModule
from src.models.eprbench_module import EPRBenchLitModule

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_TEXT = True
FEATURES_DIM = 10752
RECALL_VALUES = [1, 5, 10, 100]
INFER_BATCH_SIZE = 32
NUM_WORKERS = 2

cfg = OmegaConf.create({
    "data": {
        "_target_": "src.data.eprbench_datamodule.EPRBenchDataModule",
        "dataset_folder": str(LOCAL_EVENTVPR_DIR),
        "train_batch_size": 18,
        "infer_batch_size": INFER_BATCH_SIZE,
        "num_workers": NUM_WORKERS,
        "pin_memory": True,
        "use_text": USE_TEXT,
        "text_folder": None,
        "img_per_place": 5,
        "min_img_per_place": 5,
        "resize": [224, 224],
    },
    "model": {
        "_target_": "src.models.eprbench_module.EPRBenchLitModule",
        "lambda_contrast": 0.15,
        "temperature": 0.07,
        "use_text": USE_TEXT,
        "features_dim": FEATURES_DIM,
        "test_method": "hard_resize",
        "recall_values": RECALL_VALUES,
        "infer_batch_size": INFER_BATCH_SIZE,
        "num_workers": NUM_WORKERS,
        "pca_dim": None,
        "efficient_ram_testing": False,
        "majority_weight": 0.01,
        "compile": False,
        "optimizer": {
            "_target_": "torch.optim.AdamW",
            "_partial_": True,
            "lr": 0.0001,
            "weight_decay": 0.001,
        },
        "scheduler": {
            "_target_": "torch.optim.lr_scheduler.StepLR",
            "_partial_": True,
            "gamma": 0.5,
        },
        "net": {
            "_target_": "src.models.components.network.SGVPRNet",
            "pretrained_foundation": True,
            "foundation_model_path": str(LOCAL_MODEL_DIR / "dinov2_vitb14_pretrain.pth"),
            "use_frequency_enhance": False,
            "use_freq_topk": False,
            "mods": {
                "cross_spectral": {"enabled": False, "layer_idx": 6},
                "sacf": {"enabled": False, "variant": "original", "layer_idx": 6},
                "scfm": {"enabled": False},
                "igr": {"enabled": False},
                "lgfp": {"enabled": False},
                "fbgw": {"enabled": False},
                "bim": {"enabled": False},
                "spm": {"enabled": False},
                "event_cues": {"enabled": False},
                "event_aware_token_selector": {"enabled": False},
                "adaptive_text_topk": {"enabled": False},
                "temporal_fusion": {"enabled": False},
            },
        },
    },
})

datamodule: EPRBenchDataModule = hydra.utils.instantiate(cfg.data)
datamodule.setup("test")
test_dataset = datamodule.data_test
print(f"测试集总样本数: {len(test_dataset)} (Database: {test_dataset.database_num}, Queries: {test_dataset.queries_num})")

model: EPRBenchLitModule = hydra.utils.instantiate(cfg.model).to(device)
model.eval()

sgvpr_ckpt_path = LOCAL_MODEL_DIR / "SGVPR.pth"
ckpt = torch.load(sgvpr_ckpt_path, map_location="cpu", weights_only=False)
model_state = {k.replace("module.", ""): v for k, v in ckpt["model_state_dict"].items()}
missing, unexpected = model.net.load_state_dict(model_state, strict=False)
print(f"SGVPR.pth 加载完成 | net missing: {len(missing)}, unexpected: {len(unexpected)}")
if "text_encoder_state_dict" in ckpt:
    model.text_encoder.load_state_dict(ckpt["text_encoder_state_dict"], strict=False)

# 提取特征并检索
all_features, all_indices = [], []
with torch.no_grad():
    for batch in tqdm(datamodule.test_dataloader(), desc="Extracting test features"):
        imgs, indices, text_desc = batch
        if USE_TEXT:
            text_list = list(text_desc)
            text_vec = model.text_encoder(text_list)
            text_tokens, text_mask = model.text_encoder.encode_batch_tokens(text_list)
            feats = model.net(
                imgs.to(device),
                text=text_vec,
                text_tokens=text_tokens.to(device),
                text_mask=text_mask.to(device),
            )
        else:
            feats = model.net(imgs.to(device))
        all_features.append(feats.cpu())
        all_indices.append(indices.cpu())

features_np = torch.cat(all_features, dim=0).numpy()
indices_np = torch.cat(all_indices, dim=0).numpy()

sorted_features = np.empty((len(test_dataset), FEATURES_DIM), dtype=np.float32)
sorted_features[indices_np] = features_np
db_feats = sorted_features[: test_dataset.database_num]
q_feats = sorted_features[test_dataset.database_num :]

faiss_index = faiss.IndexFlatL2(FEATURES_DIM)
faiss_index.add(db_feats)
_, predictions = faiss_index.search(q_feats, max(RECALL_VALUES))

positives_per_query = test_dataset.get_positives()
recalls_per_query = np.zeros((len(positives_per_query), len(RECALL_VALUES)), dtype=bool)
for q_idx, (pred, pos) in enumerate(zip(predictions, positives_per_query)):
    for r_i, n in enumerate(RECALL_VALUES):
        recalls_per_query[q_idx, r_i] = np.any(np.isin(pred[:n], pos))

overall_recalls = recalls_per_query.mean(axis=0) * 100
print("\n🎯 官方 SGVPR.pth 测试集检索结果 (Expected ~ R@1: 94.0%):")
for n, rec in zip(RECALL_VALUES, overall_recalls):
    print(f"  R@{n:<3}: {rec:.2f}%")

In [ ]:
# 随机抽取一个 Query 可视化 Top-5 检索结果
TOP_K = 5
query_idx = random.randint(0, test_dataset.queries_num - 1)
query_global_idx = test_dataset.database_num + query_idx
query_path = test_dataset.all_images_paths[query_global_idx]
positives = positives_per_query[query_idx]
pred = predictions[query_idx][:TOP_K]

fig, axes = plt.subplots(1, TOP_K + 1, figsize=(3.2 * (TOP_K + 1), 3.6))
axes[0].imshow(Image.open(query_path).convert("RGB").resize((224, 224)))
axes[0].set_title(f"Query #{query_idx}\nPlace: {test_dataset._all_place_ids[query_global_idx]}")
axes[0].axis("off")

for rank, db_idx in enumerate(pred, start=1):
    db_path = test_dataset.all_images_paths[db_idx]
    is_pos = db_idx in positives
    ax = axes[rank]
    ax.imshow(Image.open(db_path).convert("RGB").resize((224, 224)))
    ax.set_title(
        f"Rank {rank} ({'GT✓' if is_pos else 'GT✗'})\nPlace: {test_dataset._all_place_ids[db_idx]}",
        color="white" if is_pos else "black",
        backgroundcolor="green" if is_pos else "#ffcccc",
    )
    ax.axis("off")

plt.tight_layout()
plt.show()

# 释放显存以便后续训练
del model, datamodule, features_np, sorted_features
torch.cuda.empty_cache()

## 5. 模型训练 (`src/train.py`)

### 5.1 快速冒烟测试 (1 Epoch Smoke Test，仅跑 10 个训练 batch 与 10 个验证 batch)
用于验证 Hydra 配置、DINOv2 + OpenCLIP 前向/反向传播、FAISS 验证与 Checkpoint 保存在 Colab 上完全畅通。

In [ ]:
!python src/train.py \
    data=eprbench model=eprbench trainer=gpu \
    data.train_batch_size=8 data.infer_batch_size=16 data.num_workers=2 \
    trainer.max_epochs=1 \
    trainer.limit_train_batches=10 trainer.limit_val_batches=20 \
    test=false \
    ~callbacks.rich_progress_bar

### 5.2 完整实验训练（Baseline / SACF-only）

- **选项 A (`MODE = "sacf_only"`)**：训练论文/实验推荐的 `experiment=eprbench_sacf_only`（30 epochs，预期 R@1 ≈ 94.2%，16GB T4 GPU 推荐使用 `train_batch_size=12` 防止 OOM）
- **选项 B (`MODE = "baseline"`)**：训练 EventVPR + Text 基线（30 epochs，预期 R@1 ≈ 93.6%）
- 训练输出默认保存在 `/content/eprbench_lightning/logs/train/runs/`，同时可开启自动同步到云盘 `logs_backup`。

In [ ]:
# 可选: "sacf_only" 或 "baseline"
MODE = "sacf_only"
MAX_EPOCHS = 30
NUM_WORKERS = 2

# 根据显存大小自动设定合适的 batch_size（T4 15GB 用 12，A100/L4 >22GB 可用 18）
vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3) if torch.cuda.is_available() else 15.0
TRAIN_BATCH_SIZE = 18 if (MODE == "baseline" or vram_gb >= 22.0) else 12
INFER_BATCH_SIZE = 16

if MODE == "sacf_only":
    cmd = [
        "python", "src/train.py",
        "experiment=eprbench_sacf_only",
        "trainer=gpu",
        f"data.train_batch_size={TRAIN_BATCH_SIZE}",
        f"data.infer_batch_size={INFER_BATCH_SIZE}",
        f"data.num_workers={NUM_WORKERS}",
        f"trainer.max_epochs={MAX_EPOCHS}",
        "~callbacks.rich_progress_bar",
    ]
else:
    cmd = [
        "python", "src/train.py",
        "data=eprbench",
        "model=eprbench",
        "trainer=gpu",
        "data.use_text=true",
        "model.use_text=true",
        "model.lambda_contrast=0.25",
        f"data.train_batch_size={TRAIN_BATCH_SIZE}",
        f"data.infer_batch_size={INFER_BATCH_SIZE}",
        f"data.num_workers={NUM_WORKERS}",
        f"trainer.max_epochs={MAX_EPOCHS}",
        "~callbacks.rich_progress_bar",
    ]

print("执行训练命令:\n ", " ".join(cmd))
subprocess.run(cmd, check=True)

## 6. 评估训练生成的 Lightning Checkpoint (`src/eval.py`) 与备份到云盘

- 自动查找 `logs/train/runs/` 或云盘 `logs_backup/` 下最新的 `.ckpt` 权重进行测试集评估
- 若 Checkpoint 使用了 SACF 训练，评估时需开启 `model.net.mods.sacf.enabled=true`

In [ ]:
# 自动搜索最新训练生成的 .ckpt 文件（如未找到本地 ckpt，则搜索云盘 logs_backup）
ckpt_candidates = sorted(
    (PROJECT_ROOT / "logs").rglob("*.ckpt"),
    key=lambda p: p.stat().st_mtime,
    reverse=True,
)
if not ckpt_candidates and GDRIVE_LOGS_BACKUP.exists():
    ckpt_candidates = sorted(
        GDRIVE_LOGS_BACKUP.rglob("*.ckpt"),
        key=lambda p: p.stat().st_mtime,
        reverse=True,
    )

if not ckpt_candidates:
    print("⚠️ 当前未找到任何 .ckpt 文件，请先运行上方训练单元格。")
else:
    latest_ckpt = ckpt_candidates[0]
    print(f"🔍 使用 Checkpoint: {latest_ckpt}")

    # 自动检测该 ckpt 是否包含 sacf 参数
    raw = torch.load(latest_ckpt, map_location="cpu", weights_only=False)
    state_keys = list(raw.get("state_dict", {}).keys())
    has_sacf = any("sacf" in k for k in state_keys)
    print(f"自动检测 SACF 模块: {'启用 (enabled=true)' if has_sacf else '未启用 (enabled=false)'}")

    eval_cmd = [
        "python", "src/eval.py",
        f"ckpt_path={latest_ckpt}",
        "data=eprbench",
        "model=eprbench",
        "trainer=gpu",
        "data.use_text=false",
        "model.use_text=false",
        f"model.net.mods.sacf.enabled={'true' if has_sacf else 'false'}",
        "data.infer_batch_size=32",
        "data.num_workers=2",
        "~callbacks.rich_progress_bar",
    ]
    print("执行评估命令:\n ", " ".join(eval_cmd))
    subprocess.run(eval_cmd, check=True)

In [ ]:
# 将本地训练日志与 Checkpoints 增量备份到 Google Drive (datasets/EPRBench/logs_backup)
local_train_runs = PROJECT_ROOT / "logs" / "train" / "runs"
if local_train_runs.exists():
    GDRIVE_LOGS_BACKUP.mkdir(parents=True, exist_ok=True)
    dst_runs = GDRIVE_LOGS_BACKUP / "train_runs"
    print(f"⏳ 正在将 {local_train_runs} 备份至云盘 {dst_runs} ...")
    shutil.copytree(local_train_runs, dst_runs, dirs_exist_ok=True)
    print(f"✅ 备份完成！云盘路径: {dst_runs}")
else:
    print("暂无本地训练日志需要备份。")